In [1]:

import argparse
import json
import logging
from functools import reduce
from pathlib import Path

import yaml
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    StringIndexerModel,
    OneHotEncoder,
    OneHotEncoderModel,
    CountVectorizer,
    CountVectorizerModel,
    VectorAssembler,
)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

from sklearn.model_selection import StratifiedGroupKFold
from xgboost.spark import SparkXGBClassifier


logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


def train_test_split(df, group, label, train_fraction, seed):
    weights = [train_fraction, 1 - train_fraction]

    df_keys = (
        df
        .groupBy(group)
        .agg(F.sort_array(F.collect_set(label)).alias("label_set"))
        .withColumn("key", F.array_join(F.col("label_set"), "_"))
        .cache()
    )

    unique_keys = [
        p[0]
        for p in df_keys.select("key").distinct().collect()
    ]

    train_groups_list = []
    test_groups_list = []

    for key in unique_keys:
        class_groups = df_keys.filter(F.col("key") == key)

        train_split, test_split = class_groups.randomSplit(
            weights,
            seed=seed
        )

        train_groups_list.append(train_split.select(group))
        test_groups_list.append(test_split.select(group))

    train_groups = reduce(
        lambda df1, df2: df1.union(df2),
        train_groups_list
    )

    test_groups = reduce(
        lambda df1, df2: df1.union(df2),
        test_groups_list
    )

    df_train = df.join(train_groups, group, "inner")
    df_test = df.join(test_groups, group, "inner")

    df_keys.unpersist()

    return df_train, df_test


def get_feature_names(pipeline_model, num_cols, cat_cols, vec_cols):
    """
    Reconstruct feature names in VectorAssembler order.
    """

    stages = pipeline_model.stages

    indexer_model = next(
        s for s in stages
        if isinstance(s, StringIndexerModel)
    )

    cv_models = {
        s.getInputCol(): s
        for s in stages
        if isinstance(s, CountVectorizerModel)
    }

    assembler_model = next(
        s for s in stages
        if isinstance(s, VectorAssembler)
    )

    # Map categorical columns to their fitted labels
    indexer_labels = {
        col: labels
        for col, labels in zip(
            indexer_model.getInputCols(),
            indexer_model.labelsArray
        )
    }

    names = []

    # Follow the actual VectorAssembler input order
    for input_col in assembler_model.getInputCols():

        if input_col in num_cols:
            names.append(input_col)

        elif input_col.endswith("_ohe"):

            cat_col = input_col.removesuffix("_ohe")

            labels = indexer_labels[cat_col]

            names.extend(
                [f"{cat_col}={lab}" for lab in labels]
            )

            # StringIndexer(handleInvalid="keep") and
            # OneHotEncoder(dropLast=False)
            names.append(f"{cat_col}=__unknown")

        elif input_col.endswith("_vec"):

            original_col = input_col.removesuffix("_vec")

            cv_model = cv_models[original_col]

            names.extend(
                [f"{original_col}={term}"
                 for term in cv_model.vocabulary]
            )

        else:
            raise ValueError(
                f"Unrecognized assembler input: {input_col}"
            )

    return names


def get_feature_importance(pipeline_model, feature_names):
    """
    Return XGBoost feature importance as a pandas DataFrame.
    """

    booster = pipeline_model.stages[-1].get_booster()

    if booster.num_features() != len(feature_names):
        raise ValueError(
            f"Model has {booster.num_features()} features "
            f"but {len(feature_names)} names were built"
        )

    keys = booster.feature_names or [
        f"f{i}" for i in range(len(feature_names))
    ]

    importance_types = [
        "gain",
        "total_gain",
        "weight",
        "cover"
    ]

    scores = {
        t: booster.get_score(importance_type=t)
        for t in importance_types
    }

    df_imp = pd.DataFrame({
        "feature": feature_names
    })

    for t in importance_types:
        df_imp[t] = [
            float(scores[t].get(k, 0.0))
            for k in keys
        ]

    total_gain = df_imp["total_gain"].sum()

    df_imp["total_gain_fraction"] = (
        df_imp["total_gain"] / total_gain
        if total_gain > 0
        else 0.0
    )

    return (
        df_imp
        .sort_values("total_gain", ascending=False)
        .reset_index(drop=True)
    )




In [2]:

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Local-XGBoost-Training")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

   

26/10/31 12:22:45 WARN Utils: Your hostname, meiying-MS-7D07 resolves to a loopback address: 127.0.1.1; using 192.168.0.9 instead (on interface enp4s0)
26/10/31 12:22:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/31 12:22:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [1]:
cat_cols = ["gender_concept_id", "race_concept_id", "ethnicity_concept_id"]
vec_cols = ["distinct_conditions_set", "distinct_drugs_set", "distinct_procedures_set"]
drop_cols = ["person_id", "admission_group", "inpatient_visit_earliest_start_datetime", "inpatient_visit_end_datetime"]
label = "next_admission_within_30_days"
group = "person_id"


df = spark.read.parquet("/home/meiying/Desktop/readmission/part-00000-63b9a8b3-518f-478f-aee8-44e30345f1b7-c000.snappy.parquet")

num_cols = [c for c in df.columns if c not in (cat_cols + vec_cols + drop_cols + [label])]
df = df.select(group, "admission_group", label, *num_cols, *cat_cols, *vec_cols)

indexer = StringIndexer(inputCols=cat_cols, outputCols=[f"{c}_idx" for c in cat_cols], handleInvalid="keep")
encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in cat_cols], outputCols=[f"{c}_ohe" for c in cat_cols], dropLast=False)

count_vec = [CountVectorizer(inputCol=c, binary=True, minDF=50, vocabSize=200, outputCol=f"{c}_vec") for c in vec_cols]

num_cols = [c for c in df.columns if c not in (cat_cols + vec_cols + drop_cols + [label])] 
assembler = VectorAssembler(
    inputCols=num_cols + [f"{c}_ohe" for c in cat_cols] + [f"{c}_vec" for c in vec_cols],
    outputCol="features",
    handleInvalid="keep"
    )

xgb = SparkXGBClassifier(
    features_col="features", 
    label_col=label, 
    num_workers=1,
    n_estimators=100, 
    tree_method="hist",
)

pipeline = Pipeline(stages=[indexer, encoder, *count_vec, assembler, xgb])
pipeline_model = pipeline.fit(df)

NameError: name 'spark' is not defined

In [ ]:
def get_feature_names(pipeline_model, df, features_col="features"):
    attrs_by_type = pipeline_model.transform(df).schema[features_col].metadata["ml_attr"]["attrs"] # dictionary of lists
    print(attrs_by_type.values())
    attrs = []
    for group in attrs_by_type.values():
        for a in group:
            attrs.append(a)
    attrs = sorted(attrs, key=lambda a: a["idx"])
    print(attrs)
    vocabs = {s.getOutputCol(): s for s in pipeline_model.stages if isinstance(s, CountVectorizerModel)}
    print(vocabs)

    names = []
    for attr in attrs:
        col, _, index = attr["name"].rpartition("_")
        if col in vocabs and index.isdigit():
            cv = vocabs[col]
            names.append(f"{cv.getInputCol()}={cv.vocabulary[int(index)]}")
        else:
            names.append(attr["name"])
    return names

def get_feature_importance(model, feature_names):
    importance_types = ["gain", "total_gain", "weight", "cover"]

    booster = model.stages[-1].get_booster().copy()
    booster.feature_names = feature_names
    scores = {t: booster.get_score(importance_type=t) for t in importance_types}
    df_imp = pd.DataFrame(scores, index=feature_names)
    df_imp = df_imp.fillna(0)
    df_imp["total_gain_fraction"] = df_imp["total_gain"] / df_imp["total_gain"].sum()
    df_imp = df_imp.sort_values(["total_gain_fraction"], ascending=False)
    # give index column a label and change index to regular column
    df_imp = df_imp.rename_axis("features").reset_index()
    return df_imp


In [ ]:

feature_names = get_feature_names(pipeline_model, df)
df_imp = get_feature_importance(pipeline_model, feature_names)
spark.createDataFrame(df_imp).coalesce(1).write.csv("feature_importance.csv", mode="overwrite", header=True)